<div style="font-family: 'Times New Roman', Times, serif;
text-align: center; padding: 120px 60px; page-break-after: always;">

<br><br><br>

<p style="font-size: 24px; font-weight: bold; letter-spacing: 1px; margin-bottom: 50px;">
Credit Portfolio Risk Analysis
</p>

<hr style="border: none; border-top: 1.5px solid black; width: 75%; margin: 6px auto;">
<hr style="border: none; border-top: 1.5px solid black; width: 75%; margin: 6px auto;">

<br><br><br>

<p style="font-size: 12px; line-height: 2.8; text-align: center;">
<b>Student:</b>&nbsp;&nbsp; Shazia Ishaq <br>
<b>Portfolios:</b>&nbsp;&nbsp; Portfolio 1 and Portfolio 2 <br>
<b>Course:</b>&nbsp;&nbsp; Introduction to Credit Risk and Applications in Python <br>
<b>Date:</b>&nbsp;&nbsp; June 2026
</p>

<br><br><br>

<hr style="border: none; border-top: 1.5px solid black; width: 75%; margin: 6px auto;">
<hr style="border: none; border-top: 1.5px solid black; width: 75%; margin: 6px auto;">

</div>


In [ ]:
import os, pickle, warnings
import pandas as pd
import numpy as np
from scipy.stats import norm
from IPython.display import display, Image
from IPython.core.display import HTML
warnings.filterwarnings('ignore')

display(HTML('''
<style>
@media print {
    .jp-Cell { page-break-inside: avoid; }
    h2 { page-break-before: always; }
    h2:first-of-type { page-break-before: avoid; }
}
body, p, li, td, th {
    font-family: "Times New Roman", Times, serif !important;
    font-size: 12pt !important;
    line-height: 1.15 !important;
}
h1 { font-size: 16pt !important; font-family: "Times New Roman", Times, serif !important; }
h2 { font-size: 14pt !important; font-family: "Times New Roman", Times, serif !important; }
h3 { font-size: 12pt !important; font-family: "Times New Roman", Times, serif !important; }
.dataframe { font-size: 10.5pt !important; width: 100% !important; }
.dataframe thead th { background-color: #1a3a5c !important; color: white !important; }
.dataframe tbody tr:nth-child(even) { background-color: #f0f4f8; }
</style>
'''))

RESULTS = os.path.join(
    os.path.dirname(os.path.abspath('__file__')),
    '..', 'results')

params = pd.read_csv(os.path.join(RESULTS, 'parameters.csv'))
par1 = params[params['portfolio']=='Portfolio 1'].dropna(
    subset=['EAD','PD','LGD','rho']).reset_index(drop=True)
par2 = params[params['portfolio']=='Portfolio 2'].dropna(
    subset=['EAD','PD','LGD','rho']).reset_index(drop=True)

np.random.seed(42)
N_SIM = 100_000
ALPHA = 0.99

def run_simulation(table):
    PDs  = table['PD'].values
    LGDs = table['LGD'].values
    EADs = table['EAD'].values
    rhos = table['rho'].values
    m    = len(table)
    w    = EADs / EADs.sum()
    d    = norm.ppf(PDs)
    F    = np.random.normal(0, 1, N_SIM)
    eps  = np.random.normal(0, 1, (N_SIM, m))
    X    = rhos * F[:, None] + np.sqrt(1 - rhos**2) * eps
    Y    = (X <= d).astype(float)
    L    = Y @ (w * LGDs)
    EL_gf  = (PDs * LGDs * w).sum()
    VaR_gf = np.percentile(L, 99)
    ES_gf  = L[L > VaR_gf].mean()
    Std_gf = L.std()
    pm    = PDs.mean()
    var_p = 0.005 * pm * (1 - pm)
    com   = pm * (1 - pm) / var_p - 1
    L_bb  = np.zeros(N_SIM)
    for s in range(N_SIM):
        P = np.random.beta(pm * com, (1 - pm) * com)
        Ybb = np.random.binomial(1, P, m).astype(float)
        L_bb[s] = (Ybb * w * LGDs).sum()
    VaR_bb = np.percentile(L_bb, 99)
    ES_bb  = L_bb[L_bb > VaR_bb].mean()
    return dict(
        EL_gf=EL_gf, Std_gf=Std_gf, VaR_gf=VaR_gf, ES_gf=ES_gf,
        EL_bb=L_bb.mean(), Std_bb=L_bb.std(),
        VaR_bb=VaR_bb, ES_bb=ES_bb,
        EAD_total=EADs.sum(), m=m,
        mean_PD=PDs.mean(), mean_LGD=LGDs.mean(),
        mean_rho=rhos.mean(), L_gf=L, L_bb=L_bb)

r1 = run_simulation(par1)
r2 = run_simulation(par2)


<div style="font-family: 'Times New Roman', Times, serif;
page-break-after: always; padding: 40px 0;">

## Contents

1. &nbsp; Introduction &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; 3
2. &nbsp; Data &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; 3
3. &nbsp; Methodology &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; 4
4. &nbsp; Results and Discussion &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; 4
5. &nbsp; Conclusion &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; 6
6. &nbsp; References &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; 7

</div>



## 1. Introduction

&nbsp;&nbsp;&nbsp;&nbsp;Credit portfolio risk quantifies potential losses from the simultaneous default of multiple obligors. Unlike single-name credit risk, portfolio credit risk depends not only on individual creditworthiness but also on the dependence structure among borrowers. When defaults are positively correlated, adverse macroeconomic conditions can trigger clustered losses far exceeding what individual default probabilities suggest in isolation (Vasicek, 1987). The Basel III Internal Ratings-Based (IRB) framework requires banks to estimate portfolio loss distributions and hold capital sufficient to absorb unexpected losses at high confidence levels (Basel Committee, 2006).

&nbsp;&nbsp;&nbsp;&nbsp;This report analyzes the one-year credit risk of two simulated corporate portfolios. Portfolio 1 consists of 28 Industry A obligors and Portfolio 2 consists of 24 Industry B obligors. Two models are implemented: the Gaussian One-Factor Model (Vasicek, 1987), which underpins the Basel III IRB framework, and the Beta-Bernoulli mixture model. Risk measures including Expected Loss (EL), Value at Risk (VaR 99%), and Expected Shortfall (ES 99%) are computed and compared across portfolios and models.



## 2. Data

### 2.1 Dataset Description

&nbsp;&nbsp;&nbsp;&nbsp;Six raw datasets were provided, all intentionally containing quality issues. The *entities_info.csv* file contains firm characteristics for 103 firms including analyst-estimated default probabilities (base_pd_hint), loss given default (lgd), industry, and region. The *portfolio_1.csv* and *portfolio_2.csv* files provide exposure at default (EAD) for 31 firms each before cleaning. The *stock_returns.csv* provides 78,320 daily equity return observations used to estimate systematic sensitivity. The *industry_index_returns.csv* contains 1,566 daily industry index returns, and *default_history_20y.csv* records annual default events over 20 years for PD validation.

### 2.2 Data Quality Issues and Cleaning

&nbsp;&nbsp;&nbsp;&nbsp;The following issues were identified and resolved. Entity codes had inconsistent case and whitespace — standardized by stripping and uppercasing. Industry labels had four variants for the same category — mapped to canonical names. Date columns mixed ISO and European formats — parsed with mixed-format detection. Missing LGD (6 values) and PD (5 values) were imputed with sample medians. Missing stock returns (2,242 rows) and EAD (2 per portfolio) were dropped. Missing default events (47) were set to zero. Duplicates were removed across all files keeping the first occurrence.

### 2.3 Assumptions

&nbsp;&nbsp;&nbsp;&nbsp;Missing LGD was filled with the median (0.471) as a robust estimate for unsecured corporate exposures. Missing PD was filled with the median (2.67%), consistent with the peer group average. Missing default events were treated as zero, assuming absence of a record implies no default. Rows with missing EAD were dropped since exposure is required for loss calculation; the two affected rows per portfolio represent under 7% of obligors. Firms with fewer than 30 return observations received the portfolio median rho as a conservative fallback.

### 2.4 Final Dataset Summary


In [ ]:
summary = pd.DataFrame({
    'Metric': [
        'Number of obligors',
        'Total EAD (EUR millions)',
        'Mean PD (%)',
        'PD range (%)',
        'Mean LGD (%)',
        'LGD range (%)',
        'Total Expected Loss (EUR M)',
        'Mean rho',
        'rho range',
        'Industry',
    ],
    'Portfolio 1': [
        str(r1['m']),
        f"{r1['EAD_total']/1e6:.1f}",
        f"{r1['mean_PD']*100:.3f}",
        f"{par1['PD'].min()*100:.3f} to {par1['PD'].max()*100:.3f}",
        f"{r1['mean_LGD']*100:.1f}",
        f"{par1['LGD'].min()*100:.1f} to {par1['LGD'].max()*100:.1f}",
        f"{(par1['PD']*par1['LGD']*par1['EAD']).sum()/1e6:.4f}",
        f"{r1['mean_rho']:.3f}",
        f"{par1['rho'].min():.3f} to {par1['rho'].max():.3f}",
        'Industry A',
    ],
    'Portfolio 2': [
        str(r2['m']),
        f"{r2['EAD_total']/1e6:.1f}",
        f"{r2['mean_PD']*100:.3f}",
        f"{par2['PD'].min()*100:.3f} to {par2['PD'].max()*100:.3f}",
        f"{r2['mean_LGD']*100:.1f}",
        f"{par2['LGD'].min()*100:.1f} to {par2['LGD'].max()*100:.1f}",
        f"{(par2['PD']*par2['LGD']*par2['EAD']).sum()/1e6:.4f}",
        f"{r2['mean_rho']:.3f}",
        f"{par2['rho'].min():.3f} to {par2['rho'].max():.3f}",
        'Industry B',
    ]
}).set_index('Metric')

print("Table 1: Dataset Summary after Cleaning")
display(summary)


## 3. Methodology

### 3.1 Gaussian One-Factor Model

&nbsp;&nbsp;&nbsp;&nbsp;The Gaussian One-Factor Model (Vasicek, 1987), adopted in the Basel III IRB framework (Basel Committee, 2006), represents each firm's creditworthiness as:

$$X_i = \rho_i \cdot F + \sqrt{1 - \rho_i^2} \cdot \varepsilon_i, \quad F, \varepsilon_i \sim N(0,1)$$

where $F$ is a common economic factor and $\rho_i$ is the systematic sensitivity. Firm $i$ defaults when $X_i \leq d_i = \Phi^{-1}(PD_i)$. Portfolio loss is $L = \sum_i Y_i \cdot w_i \cdot LGD_i$ where $w_i = EAD_i / \sum_j EAD_j$. The analytical Expected Loss is $EL = \sum_i PD_i \cdot LGD_i \cdot w_i$.

### 3.2 Beta-Bernoulli Model

&nbsp;&nbsp;&nbsp;&nbsp;The Beta-Bernoulli model (course Task 6; Fitch Ratings, 2008) treats the default probability as random: $P \sim \text{Beta}(\alpha, \beta)$, with each firm defaulting as $Y_i \mid P \sim \text{Bernoulli}(P)$. All firms share the same $P$ per scenario, creating dependence without explicit rho estimation. Parameters $\alpha$ and $\beta$ are calibrated via method of moments to match the portfolio mean PD and an assumed default correlation of 0.5%.

### 3.3 Parameter Estimation

&nbsp;&nbsp;&nbsp;&nbsp;**PD** is taken from base_pd_hint in entities_info.csv and validated against 20-year empirical default rates (Pearson correlation = 0.366, Figure 2), confirming consistency with historical experience. **rho** is estimated as the Pearson correlation between each firm's daily equity return and its industry index return, following Merton (1974), and clipped to [0.1, 0.9]. **Simulation** uses 100,000 Monte Carlo scenarios (seed=42) at 99% confidence, consistent with Basel III standards.


In [ ]:
print("Figure 1: Distribution of Estimated rho by Portfolio")
display(Image(os.path.join(RESULTS, 'rho_distribution.png'), width=750))

In [ ]:
print("Figure 2: PD Validation — base_pd_hint vs Empirical PD (correlation = 0.366)")
display(Image(os.path.join(RESULTS, 'pd_validation.png'), width=530))


## 4. Results and Discussion

### 4.1 Risk Measures


In [ ]:
results_table = pd.DataFrame({
    'Measure':           ['EL', 'Std', 'VaR 99%', 'ES 99%'],
    'P1 Gaussian':       [r1['EL_gf'],  r1['Std_gf'],
                          r1['VaR_gf'], r1['ES_gf']],
    'P1 Beta-Bernoulli': [r1['EL_bb'],  r1['Std_bb'],
                          r1['VaR_bb'], r1['ES_bb']],
    'P2 Gaussian':       [r2['EL_gf'],  r2['Std_gf'],
                          r2['VaR_gf'], r2['ES_gf']],
    'P2 Beta-Bernoulli': [r2['EL_bb'],  r2['Std_bb'],
                          r2['VaR_bb'], r2['ES_bb']],
}).set_index('Measure')

print("Table 2: Risk Measures (100,000 simulations | 99% confidence | seed=42)")
display(results_table.round(6))

print("Comparison ratios (Gaussian One-Factor):")
print("  P2/P1 EL ratio:  " + str(round(r2['EL_gf']/r1['EL_gf'],2)) + "x")
print("  P2/P1 VaR ratio: " + str(round(r2['VaR_gf']/r1['VaR_gf'],2)) + "x")
print("  P2/P1 ES ratio:  " + str(round(r2['ES_gf']/r1['ES_gf'],2)) + "x")
print("Absolute EL:")
print("  P1: EUR " + str(round(r1['EL_gf']*r1['EAD_total']/1e6,2))
      + "M (" + str(round(r1['EL_gf']*100,3)) + "% of EAD)")
print("  P2: EUR " + str(round(r2['EL_gf']*r2['EAD_total']/1e6,2))
      + "M (" + str(round(r2['EL_gf']*100,3)) + "% of EAD)")


### 4.2 Portfolio 1 vs Portfolio 2

&nbsp;&nbsp;&nbsp;&nbsp;Portfolio 2 (Industry B) exhibits substantially higher credit risk across all measures. Under the Gaussian model, Portfolio 2 yields EL of 1.65% of EAD (EUR 9.63M) versus 0.91% (EUR 5.76M) for Portfolio 1 — a 1.81x difference driven directly by higher mean PD (3.311% vs 2.034%) and LGD (49.5% vs 46.8%). The tail risk difference is even more pronounced: VaR 99% for Portfolio 2 (14.54% of EAD) exceeds Portfolio 1 (7.89%) by 1.84x, and ES 99% (18.49% vs 10.22%) by 1.81x. The large gap between EL and VaR — approximately 8–9x — reflects the highly right-skewed nature of credit loss distributions: most scenarios produce near-zero losses while rare severe downturns generate disproportionately large losses.

### 4.3 Effect of Systematic Sensitivity (rho)

&nbsp;&nbsp;&nbsp;&nbsp;Portfolio 2 firms exhibit higher rho (mean 0.540, range 0.480–0.579) than Portfolio 1 (mean 0.441, range 0.410–0.495). Since the analytical EL formula $EL = \sum_i PD_i \cdot LGD_i \cdot w_i$ does not involve rho, this difference has no direct effect on expected losses. However, higher rho increases the probability of simultaneous defaults in adverse scenarios, widening the tail without shifting the mean. This explains why the VaR and ES ratios (1.84x and 1.81x) slightly exceed the EL ratio (1.81x), a result consistent with course Tasks 4 and 5.

### 4.4 Model Comparison


In [ ]:
print("Figure 3: Simulated Loss Distributions — Gaussian vs Beta-Bernoulli")
display(Image(os.path.join(RESULTS, 'loss_distributions.png'), width=900))

In [ ]:
print("Figure 4: Full Risk Measure Comparison — Both Portfolios and Models")
display(Image(os.path.join(RESULTS, 'comparison_chart.png'), width=900))


&nbsp;&nbsp;&nbsp;&nbsp;Both models produce closely aligned EL estimates — Portfolio 1: 0.9128% (Gaussian) vs 0.9307% (Beta-Bernoulli); Portfolio 2: 1.6506% vs 1.5981% — confirming that EL depends only on PD, LGD, and EAD regardless of dependence structure. The models diverge substantially in tail risk. For Portfolio 1, Gaussian VaR (7.89%) exceeds Beta-Bernoulli VaR (5.65%) by 40%, and ES by 51%. For Portfolio 2 the differences are 82% and 97% respectively. The Gaussian model produces higher tail risk because firm-specific rho values (mean 0.441–0.540) translate into strong systematic dependence, while the Beta-Bernoulli model uses only 0.5% default correlation. This model risk — capital estimates varying by nearly a factor of two — illustrates why implementing multiple models is valuable for robust risk assessment. The Fitch Ratings (2008) report notes that empirically implied correlations are often lower than Basel II regulatory values, consistent with our finding that equity-return based rho estimates exceed Basel regulatory thresholds.



## 5. Conclusion

&nbsp;&nbsp;&nbsp;&nbsp;This report quantified the one-year credit risk of two simulated corporate portfolios using the Gaussian One-Factor Model and the Beta-Bernoulli model. Three principal findings emerge. First, Portfolio 2 (Industry B) exhibits materially higher risk across all measures — EL 1.81x, VaR 99% 1.84x, and ES 99% 1.81x higher than Portfolio 1 — driven by higher PD, LGD, and systematic sensitivity. Second, both models agree on EL but diverge substantially in tail risk measures (40–97%), illustrating the sensitivity of capital estimates to dependence model choice. Third, ES/VaR ratios of 1.27–1.30x confirm heavy-tailed loss distributions, underscoring ES as the preferred regulatory tail risk measure.

&nbsp;&nbsp;&nbsp;&nbsp;Limitations include the short rho estimation window (approximately two years), which may understate stress-period correlations (Fitch Ratings, 2008), and reliance on analyst PD estimates that may not reflect current market conditions. Future work could incorporate stress-tested rho scenarios and time-varying PD models calibrated to credit spreads.



<div style="page-break-before: always; font-family: 'Times New Roman', Times, serif;">

## References

Basel Committee on Banking Supervision. (2006). *International Convergence of Capital Measurement and Capital Standards: A Revised Framework*. Bank for International Settlements. https://www.bis.org/publ/bcbs128.htm

Fitch Ratings. (2008). *Understanding Fitch's Empirical Approach to Basel II Correlation Values*. Fitch Ratings Special Report.

Merton, R. C. (1974). On the pricing of corporate debt: The risk structure of interest rates. *Journal of Finance*, *29*(2), 449–470. https://doi.org/10.1111/j.1540-6261.1974.tb03058.x

Vasicek, O. (1987). *Probability of Loss on Loan Portfolio*. KMV Corporation Working Paper.

</div>
